In [28]:
import polars as pl
import pandas as pd

## Загрузка 2.0

In [23]:
pddf = pd.read_excel('df/data_marker_1_2020.xlsx',
                     sheet_name="Результаты поиска",
                     dtype={'Реестровый номер публикации': str,
                            'ИНН заказчика': str,
                            'ИНН поставщика': str})

In [30]:
df = pl.from_pandas(pddf)

In [39]:
dforg = df

In [40]:
df = df.drop(['Банковское \\ казначейское сопровождение', 'Источник финансирования', 'Ссылка на источник'])

In [68]:
for i in range(4, 16):
    new = pd.read_excel(f'df/data_marker_{i}_2020.xlsx',
                        sheet_name="Результаты поиска",
                        dtype={'Реестровый номер публикации': str,
                                'ИНН заказчика': str,
                                'ИНН поставщика': str})
    newpl = pl.from_pandas(new)
    newpl = newpl.drop(['Банковское \\ казначейское сопровождение', 'Источник финансирования', 'Ссылка на источник'])
    print(i, new.shape, newpl.shape)
    
    newpl = newpl.with_columns(pl.col('Идентификационный код закупки').cast(pl.String))
    newpl = newpl.with_columns(pl.col('Дата  окончания проведения торгов').cast(pl.Datetime('ns')))


    df = pl.concat([df, newpl])
    

4 (808171, 31) (808171, 28)
5 (752787, 31) (752787, 28)
6 (763198, 31) (763198, 28)
7 (803087, 31) (803087, 28)
8 (648139, 31) (648139, 28)
9 (882757, 31) (882757, 28)
10 (814251, 31) (814251, 28)
11 (858080, 31) (858080, 28)
12 (897081, 31) (897081, 28)
13 (785894, 31) (785894, 28)
14 (960160, 31) (960160, 28)
15 (953149, 31) (953149, 28)


In [69]:
loaded_df = df

In [71]:
df.shape

(12406878, 28)

2 (1083717, 31) (1083717, 28) 

3 (927562, 31) (927562, 28)

### Обработка

In [74]:
data = df.rename({'Стоимость \n(руб.) Заказчик' : 'Стоимость(руб.) Заказчик',
                  'Дата \nокончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ':'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
                  'Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ' : 'Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ',
                  'Дата  окончания проведения торгов' : 'Дата окончания проведения торгов',
                  'Стоимость\n(руб.) Поставщик' : 'Стоимость(руб.) Поставщик'})

In [77]:
data.columns

['Уровень',
 'Заказчик',
 'ИНН заказчика',
 'Стоимость(руб.) Заказчик',
 'Реестровый номер публикации',
 'Идентификационный код закупки',
 'Сфера деятельности',
 'Наименование публикации',
 'Регион поставки',
 'Город поставки',
 'Дата публикации',
 'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
 'Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ',
 'Дата окончания проведения торгов',
 'Поставщик',
 'ИНН поставщика',
 'Победитель',
 'Статус допуска',
 'Стоимость(руб.) Поставщик',
 'Снижение на торгах,%',
 'Форма публикации',
 'Тип торгов',
 'Торговая площадка',
 'Электронные торги',
 'Обеспечение заявки (руб.)',
 'Обеспечение заявки, %',
 'Обеспечение контракта (руб.)',
 'Обеспечение контракта, %']

In [78]:
data = data.drop(['Наименование публикации', 'Стоимость(руб.) Поставщик', 'Торговая площадка', 'Электронные торги'])

In [79]:
data.head(2)

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %"
i64,str,str,f64,str,str,str,str,str,datetime[ns],datetime[ns],datetime[ns],datetime[ns],str,str,str,str,f64,str,str,f64,f64,f64,f64
1,"""УФССП РОССИИ ПО УЛЬЯНОВСКОЙ ОБ…","""7327033261""",123475.0,"""100270018220000024""",null,"""Неизвестно""","""Ульяновская область""","""Ульяновск""",2020-12-14 06:53:37,2020-01-31 23:59:59,null,null,"""ООО ""Л-КАРД""""","""7327065672""",null,null,null,"""Контракт""","""Иной способ""",null,null,null,null
1,"""УФССП РОССИИ ПО УЛЬЯНОВСКОЙ ОБ…","""7327033261""",3150.0,"""100270018220000026""",null,"""Неизвестно""","""Ульяновская область""","""Ульяновск""",2020-12-22 15:42:33,2020-01-31 23:59:59,null,null,"""АНО СПОРТКЛУБ ""ДИНАМО""""","""7303025600""",null,null,null,"""Контракт""","""Иной способ""",null,null,null,null


Оставляем только те строки, где есть ИНН заказчика и поставщика.

In [80]:
data = data.filter(pl.col("ИНН поставщика").is_not_null() & pl.col("ИНН заказчика").is_not_null())
data.shape

(538277, 24)

Оставляем только те строки, где известно снижение на торгах

In [81]:
data = data.filter(pl.col("Снижение на торгах,%").is_not_null())
data.shape

(457531, 24)

Уберем нереалистичные значения в ставках в Снижение на торгах,% 

In [82]:
data = data.filter((pl.col('Снижение на торгах,%')>=-100))
data.shape

(457028, 24)

Теперь работаем с датами

In [83]:
data = data.filter(pl.col('Дата публикации').is_not_null() & 
                    pl.col('Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ').is_not_null() & 
                    pl.col('Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ').is_not_null() & 
                    pl.col('Дата окончания проведения торгов').is_not_null())
data.shape

(410446, 24)

Убираем дупликаты

In [84]:
data = data.unique()
data.shape

(404863, 24)

Смотрим, по каким данным определять уникальность закупки

In [85]:
data.filter(pl.col('Реестровый номер публикации')!=pl.lit("")).shape

(404863, 24)

In [94]:
data.unique(['ИНН заказчика', 'Реестровый номер публикации']).shape

(199241, 25)

In [95]:
data.unique(['ИНН заказчика', 
             'Реестровый номер публикации', 
             'Дата публикации',
             'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
             'Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ',
             'Дата окончания проведения торгов']).shape

(199248, 25)

Отбираем тендеры, где известен победитель

In [86]:
data.select(pl.all().null_count())

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %"
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,246547,0,371053,14,0,0,0,0,0,0,0,0,339368,0,0,0,276884,380755,391940,390810,374778


In [87]:
data = data.with_columns(pl.when(pl.col('Победитель')==pl.lit('Победитель')).then(1).otherwise(0).alias('Есть победитель'))
data.shape

(404863, 25)

In [96]:
data_group = data.group_by(['Реестровый номер публикации']).agg(pl.col('Есть победитель').sum().alias('Num winner'))
data_group.shape

(198379, 2)

In [97]:
data_new = data.join(data_group,
                     how='left',
                     on=['Реестровый номер публикации'])
data_new.shape

(404863, 26)

In [99]:
data = data_new.filter(pl.col('Num winner')>=1)
data.shape

(109775, 26)

In [100]:
data = data.drop(["Есть победитель", "Num winner"])

Добавляем реестр недобросовестных поставщиков

In [105]:
notrust = pl.read_json('notrust_list/res.json')

In [106]:
notrust = notrust.rename({'suplier_name':'Поставщик', 'suplier_inn':'ИНН поставщика', 'no_trust_now':'РНП сейчас', 'no_trust_before':'РНП ранее'})

In [107]:
data = data.join(other=notrust.select(['ИНН поставщика', 'РНП сейчас', 'РНП ранее']), on='ИНН поставщика', how='left')
data = data.with_columns(pl.col('РНП сейчас').fill_null(0), pl.col('РНП ранее').fill_null(0))

In [110]:
data.filter(pl.col('РНП ранее')==1).shape

(419, 26)

Записываем результат

In [111]:
data.shape

(109900, 26)

In [112]:
data.write_excel( dtype_formats={pl.Date: "yyyy-mm-dd"})

--------

In [4]:
import polars as pl

In [21]:
df = pl.read_excel(source='/Users/renatavaliullina/Desktop/rec b2b/data_final(2020-2025)/data_final_res(2020-2025)_revenue_no_null.xlsx')
df = df.select(['ИНН заказчика', 'Стоимость(руб.) Заказчик', 'Реестровый номер публикации'])
df = df.rename({'ИНН заказчика':'inn', 'Стоимость(руб.) Заказчик':'price', 'Реестровый номер публикации':'num'})
df = df.with_columns(pl.col('price').floor().cast(pl.Int64).alias('floor'), pl.col('price').ceil().cast(pl.Int64).alias('ceil'))
df = df.unique().sort('inn')
#df = df.to_pandas()

In [43]:
import re

s = "\n    \n        \n        \n            15 %\n        \n    \n"
num = re.search(r'(\d+)\s*%', s)  
num.group(1)

'15'

In [63]:
s = "\n    \n        \n            24 000,00 ₽\n            \n                (20 %)\n            \n        \n        \n    \n"
s.split()

['24', '000,00', '₽', '(20', '%)']

------------------

In [2]:
import polars as pl

In [3]:
df = pl.read_excel('correct_data/correct_data_2020.xlsx')
df.head(2)

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
2,"""ГБУЗ АО ""ГКБ №3""""","""3018005693""",49120.0,"""0325500000120000008""","""20-23018005693302301001-0002-0…","""[ОКПД2 01.47] Птица сельскохоз…","""Астраханская область""","""Астрахань""",2020-01-17 11:42:46,2020-01-27 09:00:00,2020-01-17 13:42:45,2020-01-28 00:00:00,"""Полянский Владимир Владимирови…","""301501912801""","""Победитель""","""Допущен""",0.397394,"""Торговая процедура""",null,null,null,null,0.05,0,0
1,"""ФГБУ ""ВИМС""""","""7706433263""",420000.0,"""32009380421""",null,"""[ОКПД2 29.20] Кузова (корпуса)…","""Иркутская область""","""Иркутская область""",2020-08-05 13:07:54,2020-08-05 13:20:00,2020-08-05 00:00:00,2020-08-05 00:00:00,"""ООО ""ТК ""ЕВА""""","""3810065555""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0


In [10]:
df.filter(pl.col('ИНН заказчика')=='0274157821').filter(pl.col('Реестровый номер публикации')=='0801200000220000066').head(12)

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
2,"""ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ""","""0274157821""",374867.36,"""0801200000220000066""","""20-20274157821027401001-0191-0…","""Неизвестно""","""Республика Башкортостан""","""Уфа""",2020-01-23 16:40:02,2020-01-31 09:00:00,2020-01-23 18:27:48,2020-02-03 00:00:00,"""ООО ""АЛЬБАТРОС""""","""7724922443""",null,"""Допущен""",0.05,"""Торговая процедура""",null,3748.67,0.01,37486.74,0.1,0,0
2,"""ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ""","""0274157821""",374867.36,"""0801200000220000066""","""20-20274157821027401001-0191-0…","""Неизвестно""","""Республика Башкортостан""","""Уфа""",2020-01-23 16:40:02,2020-01-31 09:00:00,2020-01-23 18:27:48,2020-02-03 00:00:00,"""АО ""ЛАНЦЕТ""""","""7718538045""","""Победитель""","""Допущен""",0.055,"""Торговая процедура""",null,3748.67,0.01,37486.74,0.1,0,0


In [2]:
import pandas as pd

In [27]:
pl.from_pandas(t)

Уровень,Заказчик,ИНН заказчика,Стоимость (руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Наименование публикации,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок/Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,Стоимость (руб.) Поставщик,"Снижение на торгах,%",Форма публикации,Тип торгов,Торговая площадка,Электронные торги,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",Банковское \ казначейское сопровождение,Источник финансирования,Ссылка на источник
i64,str,str,f64,str,str,str,str,str,str,datetime[ns],datetime[ns],datetime[ns],datetime[ns],str,str,str,str,f64,f64,str,str,str,str,f64,f64,f64,f64,str,str,str
1,"""ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ""","""0274157821""",374867.36,"""0801200000220000066""","""20-20274157821027401001-0191-0…","""Неизвестно""","""Поставка лекарственного(-ых) п…","""Республика Башкортостан""","""Уфа""",2020-01-23 16:40:02,2020-01-31 09:00:00,2020-01-23 18:27:48,2020-02-03 00:00:00,"""Несколько поставщиков""",null,null,null,null,null,"""Торговая процедура""","""Аукцион электронный""","""ЭТС""","""Неизвестно""",3748.67,null,37486.74,null,"""Неизвестно""","""Бюджет Республики Башкортостан""","""Госзакупки 44ФЗ/94ФЗ"""
2,"""ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ""","""0274157821""",374867.36,"""0801200000220000066""","""20-20274157821027401001-0191-0…","""Неизвестно""","""Поставка лекарственного(-ых) п…","""Республика Башкортостан""","""Уфа""",2020-01-23 16:40:02,2020-01-31 09:00:00,2020-01-23 18:27:48,2020-02-03 00:00:00,"""ООО ""АЛЬБАТРОС""""","""7724922443""",null,"""Допущен""",356123.96,0.05,"""Торговая процедура""",null,null,"""Неизвестно""",3748.67,0.01,37486.74,0.1,"""Неизвестно""",null,"""Госзакупки 44ФЗ/94ФЗ"""
2,"""ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ""","""0274157821""",374867.36,"""0801200000220000066""","""20-20274157821027401001-0191-0…","""Неизвестно""","""Поставка лекарственного(-ых) п…","""Республика Башкортостан""","""Уфа""",2020-01-23 16:40:02,2020-01-31 09:00:00,2020-01-23 18:27:48,2020-02-03 00:00:00,"""АО ""ЛАНЦЕТ""""","""7718538045""","""Победитель""","""Допущен""",354249.62,0.055,"""Торговая процедура""",null,null,"""Неизвестно""",3748.67,0.01,37486.74,0.1,"""Неизвестно""",null,"""Госзакупки 44ФЗ/94ФЗ"""


In [26]:
t = pddf.loc[pddf['ИНН заказчика']=='0274157821'].loc[chunks['Стоимость \n(руб.) Заказчик']==374867.36]
t

,Уровень,Заказчик,ИНН заказчика,Стоимость \n(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Наименование публикации,Регион поставки,Город поставки,...,Тип торгов,Торговая площадка,Электронные торги,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",Банковское \ казначейское сопровождение,Источник финансирования,Ссылка на источник
40461,1,ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ,0274157821,374867.36,0801200000220000066,20-20274157821027401001-0191-001-2120-323,Неизвестно,Поставка лекарственного(-ых) препарата(-ов) дл...,Республика Башкортостан,Уфа,...,Аукцион электронный,ЭТС,Неизвестно,3748.67,NaN,37486.74,NaN,Неизвестно,Бюджет Республики Башкортостан,Госзакупки 44ФЗ/94ФЗ
40462,2,ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ,0274157821,374867.36,0801200000220000066,20-20274157821027401001-0191-001-2120-323,Неизвестно,Поставка лекарственного(-ых) препарата(-ов) дл...,Республика Башкортостан,Уфа,...,NaN,NaN,Неизвестно,3748.67,0.01,37486.74,0.1,Неизвестно,NaN,Госзакупки 44ФЗ/94ФЗ
40463,2,ГКУ ТЕХОБЕСПЕЧЕНИЕ МЗ РБ,0274157821,374867.36,0801200000220000066,20-20274157821027401001-0191-001-2120-323,Неизвестно,Поставка лекарственного(-ых) препарата(-ов) дл...,Республика Башкортостан,Уфа,...,NaN,NaN,Неизвестно,3748.67,0.01,37486.74,0.1,Неизвестно,NaN,Госзакупки 44ФЗ/94ФЗ


In [21]:
chunks.dtypes

Уровень                                                                                                                           int64
Заказчик                                                                                                                         object
ИНН заказчика                                                                                                                   float64
Стоимость \n(руб.) Заказчик                                                                                                     float64
Реестровый номер публикации                                                                                                      object
Идентификационный код закупки                                                                                                    object
Сфера деятельности                                                                                                               object
Наименование публикации                         